In [ ]:
!source myenv/bin/activate

In [ ]:
!pip install -q opencv-python matplotlib numpy pillow accelerate bitsandbytes

In [ ]:
!pip install -q accelerate

In [ ]:
!python -m pip install --upgrade --force-reinstall git+https://github.com/huggingface/transformers

In [ ]:
%%bash
git lfs install
git clone https://huggingface.co/datasets/xiang709/VRSBench

In [ ]:
import zipfile
for z in [ "VRSBench/Annotations_val.zip","VRSBench/Images_val.zip"]:
    with zipfile.ZipFile(z, 'r') as zip_ref:
        zip_ref.extractall("VRSBench_val")

In [1]:
LOCAL_ANNOTATION_DIR = "VRSBench_val/Annotations_val"
LOCAL_IMAGE_DIR = "VRSBench_val/Images_val"

In [1]:
import os
from huggingface_hub import hf_hub_download, login
import sys
import torch
login()

In [2]:
import cv2
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.ops import box_convert
import json
import glob
import re
from transformers.modeling_utils import ModuleUtilsMixin
from PIL import Image, ImageDraw, ImageFont
try:
    from transformers import (
        AutoProcessor, 
        AutoModelForZeroShotObjectDetection,
        Sam3Model, 
        Sam3Processor,
        Qwen3VLForConditionalGeneration,
        AutoModelForVision2Seq, 
        BitsAndBytesConfig
    )
    print("Loaded Transformers Classes")
    
except ImportError as e:
    raise ImportError(f"Transformers import failed: {e}. Please run the installation cell again.")

Loaded Transformers Classes


In [7]:
from torch.utils.data import Dataset
from PIL import Image

ANNOTATION_DIR = "VRSBench_val/Annotations_val"
IMAGE_DIR = "VRSBench_val/Images_val"
OUTPUT_DIR = "eval_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

class VRSBenchDataset:
    def __init__(self, anno_dir, img_dir, max_samples=5):
        self.files = glob.glob(os.path.join(anno_dir, "*.json"))[:max_samples]
        self.img_dir = img_dir
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        with open(self.files[idx], 'r') as f: data = json.load(f)
        img_name = data.get('image') or os.path.basename(self.files[idx]).replace('.json','.png')
        
        # Fix extensions logic
        img_path = os.path.join(self.img_dir, img_name)
        if not os.path.exists(img_path): 
            alt_path = img_path.replace('.png','.jpg')
            if os.path.exists(alt_path): img_path = alt_path
        
        objs = data.get('objects', [])
        return {"image_path": img_path, "image_id": img_name, "label": objs[0]['obj_cls'] if objs else None}

dataset = VRSBenchDataset(ANNOTATION_DIR, IMAGE_DIR,2)

In [ ]:
class IntegratedVLMGrounding:
    def __init__(self, device="cuda"):
        self.device = device
        print(f"--- Initializing Integrated Pipeline on {torch.cuda.get_device_name(0)} ---")
        
        print("Loading Qwen2.5-VL-3B (bfloat16)...")
        self.qwen_processor = AutoProcessor.from_pretrained(
            "Qwen/Qwen2.5-VL-3B-Instruct", 
            trust_remote_code=True
        )
        
        self.qwen_model = AutoModelForVision2Seq.from_pretrained(
            "Qwen/Qwen2.5-VL-3B-Instruct",
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.bfloat16  # Optimized for A6000
        )

        print("Loading SAM 3...")
        try:
            self.sam_processor = Sam3Processor.from_pretrained("facebook/sam3")
            self.sam_model = Sam3Model.from_pretrained("facebook/sam3").to(self.device)
        except OSError:
            print("Warning: 'facebook/sam3' not found on HF. Please ensure you have access or check the model ID.")
            raise
        
        print("Pipeline Ready.")

    def get_obb_from_mask(self, mask):
        mask_np = mask.cpu().numpy().astype(np.uint8)
        contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if len(contours) == 0:
            return None
        largest_contour = max(contours, key=cv2.contourArea)
        # minAreaRect returns ((center_x, center_y), (width, height), angle)
        obb = cv2.minAreaRect(largest_contour) 
        return obb

    def annotate_image_with_boxes(self, image, obbs):

        annotated_img = image.copy()
        draw = ImageDraw.Draw(annotated_img)

        try:
            font = ImageFont.truetype("arial.ttf", size=24)
        except:
            font = ImageFont.load_default()

        valid_candidates = {} # Map ID -> OBB
        
        for idx, obb in enumerate(obbs,start=1):
            if obb is None: continue

            box_points = cv2.boxPoints(obb)
            box_points = np.int32(box_points)
            polygon = [tuple(pt) for pt in box_points]

            draw.polygon(polygon, outline="red", width=4)

            center_x, center_y = obb[0]
            text = str(idx)
            bbox = draw.textbbox((center_x, center_y), text, font=font)
            # Expand background slightly
            draw.rectangle((bbox[0]-2, bbox[1]-2, bbox[2]+2, bbox[3]+2), fill="white")
            draw.text((center_x, center_y), text, fill="black", font=font, anchor="mm")
            
            valid_candidates[idx] = obb
            
        return annotated_img, valid_candidates

    def ask_qwen(self, image, prompt, max_tokens=128):
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        
        text_prompt = self.qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        inputs = self.qwen_processor(
            text=[text_prompt], 
            images=[image], 
            return_tensors="pt", 
            padding=True
        )
        
        inputs = inputs.to(self.qwen_model.device)
        
        output_ids = self.qwen_model.generate(**inputs, max_new_tokens=max_tokens)
 
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, output_ids)]
        output_text = self.qwen_processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)[0]
        return output_text

    def process_query(self, image_path, complex_query):
        try:
            original_image = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            return None, None

        extraction_prompt = (
            f"Identify the single main object category being referred to in this query: '{complex_query}'. "
            "For example, if the query is 'pick the red car', output 'car'. "
            "If the query is 'find the plane on the far left', output 'plane'. "
            "Output ONLY the single word category."
        )
        target_class = self.ask_qwen(original_image, extraction_prompt, max_tokens=10).strip().lower()
        target_class = re.sub(r'[^\w\s]', '', target_class).strip()
        print(f"1. Extracted Target Class: '{target_class}'")

        print(f"2. Running SAM 3 to segment all '{target_class}' instances...")
        inputs = self.sam_processor(
            images=original_image, 
            text=target_class, 
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            outputs = self.sam_model(**inputs)

        results = self.sam_processor.post_process_instance_segmentation(
            outputs, 
            threshold=0.4, 
            target_sizes=[(original_image.height, original_image.width)]
        )[0]

        masks = results["masks"] # [N, H, W]
        
        if len(masks) == 0:
            print(f"   No objects of type '{target_class}' found.")
            return None, original_image


        obbs = [self.get_obb_from_mask(m) for m in masks]
        
        annotated_image, candidates = self.annotate_image_with_boxes(original_image, obbs)

        annotated_image.save("debug_candidates.jpg")
        print(f"   Found {len(candidates)} candidates. Annotated image saved to 'debug_candidates.jpg'.")

        selection_prompt = (
            f"The user wants to: '{complex_query}'. "
            f"I have detected multiple {target_class}s and marked them with red boxes and unique numeric IDs. "
            "Based on the visual evidence, which ID number best corresponds to the user's request? "
            "Output ONLY the ID number."
        )
        
        final_answer = self.ask_qwen(annotated_image, selection_prompt, max_tokens=10)
        print(f"3. Qwen Selection Response: {final_answer}")
        
        match = re.search(r'\d+', final_answer)
        if match:
            selected_id = int(match.group())
            if selected_id in candidates:
                print(f"   SUCCESS: Selected Object ID {selected_id}")
                return candidates[selected_id], annotated_image
            else:
                print(f"   FAIL: Qwen predicted ID {selected_id}, but it is not in the valid candidate list.")
        else:
            print("FAIL: Could not parse a valid ID number from Qwen's response.")
            
        return None, annotated_image

#pipeline = IntegratedVLMGrounding(device="cuda")

In [5]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 19.2 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [9]:
class IntegratedVLMGroundingThinking:
    def __init__(self, device="cuda"):
        self.device = device
        print(f"Initializing Qwen3-VL 8B Thinking Pipeline on {torch.cuda.get_device_name(0)}")
        self.qwen_model_id = "Qwen/Qwen3-VL-8B-Thinking"
        
        # Load Qwen
        self.qwen_processor = AutoProcessor.from_pretrained(
            self.qwen_model_id, 
            trust_remote_code=True
        )
        self.qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
            self.qwen_model_id,
            
            device_map="auto",
            trust_remote_code=True,
            dtype="auto"
        )

        print("Loading SAM 3...")
        self.sam_processor = Sam3Processor.from_pretrained("facebook/sam3")
        self.sam_model = Sam3Model.from_pretrained("facebook/sam3").to(self.device)
        print("Pipeline Ready.")

    def get_obb_from_mask(self, mask):
        mask_np = mask.cpu().numpy().astype(np.uint8)
        contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if len(contours) == 0:
            return None
        largest_contour = max(contours, key=cv2.contourArea)
        # minAreaRect returns ((center_x, center_y), (width, height), angle)
        obb = cv2.minAreaRect(largest_contour) 
        return obb

    def ask_qwen(self, image, prompt, max_tokens=2048):
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        text_prompt = self.qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.qwen_processor(text=[text_prompt], images=[image], return_tensors="pt", padding=True).to(self.qwen_model.device)
        output_ids = self.qwen_model.generate(**inputs, max_new_tokens=max_tokens)
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, output_ids)]
        output_text = self.qwen_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return output_text

    def parse_thinking_output(self, text):
        clean_text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
        return clean_text if clean_text else text

    '''

    def extract_target_class_heuristic(self, query):
        stop_words = {
            "find", "locate", "identify", "show", "me", "the", "a", "an", "is", "are", 
            "where", "located", "positioned", "in", "on", "at", "of", "image", "picture", 
            "photo", "looking", "for", "object", "item", "visible"
        }
        
        target = "object"
    
        try:
            import spacy
            if not hasattr(self, 'nlp'):
                self.nlp = spacy.load("en_core_web_sm")
            
            doc = self.nlp(query)
            
            found_noun = None
        
            for token in doc:
                if token.dep_ == "dobj":
                    phrase_tokens = [t.text for t in token.subtree if t.pos_ in ["ADJ", "NOUN", "PROPN", "NUM"]]
                    if phrase_tokens:
                        found_noun = " ".join(phrase_tokens)
                        break
            if not found_noun:
                for token in doc:
                    if token.dep_ == "ROOT" and token.pos_ in ["NOUN", "PROPN"]:
                        phrase_tokens = [t.text for t in token.subtree if t.pos_ in ["ADJ", "NOUN", "PROPN"]]
                        found_noun = " ".join(phrase_tokens)
                        break
            
            if found_noun:
                target = found_noun

        except Exception as e:
            print(f"Spacy heuristic failed ({e}). using simple fallback.")
            pass

        if target == "object":
            clean_text = re.sub(r'[^\w\s]', '', query).lower()
            words = clean_text.split()
            keywords = [w for w in words if w not in stop_words]
            
            if keywords:
                target = " ".join(keywords[-2:])
                
        return target
    '''

    def process_query(self, image_path, complex_query, image_id="", known_category=None):
        try:
            original_image = Image.open(image_path).convert("RGB")
            img_w, img_h = original_image.size
        except Exception as e:
            return {"image_id": image_id, "error": f"Image Load Error: {e}", "raw_response": None}

        '''
        # 1. Extract Object Category
        
        extraction_prompt = (
            f"Analyze the query: '{complex_query}'. "
            "Identify the single main object category to find. "
            "Output ONLY the category name."
        )
        raw_class_response = self.ask_qwen(original_image, extraction_prompt, max_tokens=128)
        
        # Clean extracted class
        clean_text = re.sub(r'<think>.*?</think>', '', raw_class_response, flags=re.DOTALL)
        clean_text = re.sub(r'[^\w\s]', '', clean_text).strip()
        target_class = clean_text.split()[-1] if clean_text else "object"
        '''
        if known_category:
            target_class = known_category.strip().lower()
        else:
            target_class = self.extract_target_class_heuristic(complex_query)

            
        # Run SAM 3
        try:
            inputs = self.sam_processor(images=original_image, text=target_class, return_tensors="pt").to(self.device)
            with torch.no_grad():
                outputs = self.sam_model(**inputs)
        except Exception as e:
            return {"image_id": image_id, "error": f"SAM Error: {e}", "raw_response": None}
        
        results = self.sam_processor.post_process_instance_segmentation(
            outputs, threshold=0.3, target_sizes=[(original_image.height, original_image.width)]
        )[0]

        masks = results["masks"]
        if len(masks) == 0:
            return {"image_id": image_id, "error": "No objects detected", "raw_response": None}

        # 3. Prepare Candidates & VISUALIZE THEM
        candidates = {}
        candidate_text_lines = []
        valid_ids = []
        target_res = 512.0
        
        vis_img = np.array(original_image)
        vis_img = cv2.cvtColor(vis_img, cv2.COLOR_RGB2BGR) # Convert PIL to OpenCV BGR
        
        for idx, mask in enumerate(masks, start=1):
            obb = self.get_obb_from_mask(mask)
            if obb is None: continue
            
            (cx_real, cy_real), (w_real, h_real), angle = obb
            candidates[idx] = obb
            valid_ids.append(str(idx))

            '''
            cx_512 = (cx_real / w_real) * target_res
            cy_512 = (cy_real / h_real) * target_res
            w_512 = (w_real / w_real) * target_res
            h_512 = (h_real / h_real) * target_res
            '''
            area = w_real * h_real
            
            
            # Format text for Qwen
            desc = (f"ID {idx}: "
                    f"Center=({cx_real:.0f}, {cy_real:.0f}), "
                    f"Size=({w_real:.0f}w x {h_real:.0f}h), "
                    f"Area={area:.0f}, "
                    f"Angle={angle:.1f}")
            candidate_text_lines.append(desc)
            box_points = cv2.boxPoints(obb)
            box_points = np.int32(box_points)
            
            cv2.drawContours(vis_img, [box_points], 0, (0, 255, 0), 2)
            text_pos = (int(cx_real), int(cy_real))
            cv2.putText(vis_img, str(idx), text_pos, 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            
        candidate_text_block = "\n".join(candidate_text_lines)
        debug_dir = "debug_candidates"
        if not os.path.exists(debug_dir):
            os.makedirs(debug_dir)
            
        save_path = os.path.join(debug_dir, f"{image_id}_candidates.jpg")
        cv2.imwrite(save_path, vis_img)


        # 4. Final Prediction Prompt (remains the same)
        selection_prompt = (
            f"The user query is: '{complex_query}'.\n\n"
            f"I have mapped the image to a {int(target_res)}x{int(target_res)} coordinate grid.\n"
            f"COORDINATE SYSTEM CONVENTION:\n"
            f"- Origin (0,0) is the TOP-LEFT corner.\n"
            f"- X increases to the RIGHT (Max X={int(target_res)}).\n"
            f"- Y increases DOWNWARDS (Max Y={int(target_res)}).\n"
            f"- Center of the image is approx ({int(target_res/2)}, {int(target_res/2)}).\n\n"
            f"Here are the detected '{target_class}' candidates:\n"
            f"{candidate_text_block}\n\n"
            f"Identify the ID that matches the query based on location and size (Area). "
            f"Think step-by-step. Available IDs: {', '.join(valid_ids)}.\n"
            f"Final Answer: <ID>"
        )
        
        raw_selection_response = self.ask_qwen(original_image, selection_prompt, max_tokens=1024)
        final_answer_text = re.sub(r'<think>.*?</think>', '', raw_selection_response, flags=re.DOTALL).strip()
        match = re.search(r'\d+', final_answer_text)

        if not match:
             all_nums = re.findall(r'\d+', final_answer_text)
             if all_nums:
                 match = re.match(r'(\d+)', all_nums[-1])
        
        result_data = {
            "image_id": image_id,
            "query": complex_query,
            "target_class": target_class,
            "predicted_id": None,
            "predicted_obb": None, 
            "raw_response": raw_selection_response,
            "debug_image_path": save_path, # Save the path so you know where to look
            "selection_prompt": selection_prompt
        }

        if match:
            try:
                selected_id = int(match.group(1)) if len(match.groups()) > 0 else int(match.group(0))
                if selected_id in candidates:
                    result_data["predicted_id"] = selected_id
                    # RETRIEVE REAL COORDINATES (Not the 512 ones)
                    # We use the 'candidates' dict which stored the original OBB
                    real_obb = candidates[selected_id]
                    box_points = cv2.boxPoints(real_obb) 
                    result_data["predicted_obb"] = box_points.tolist()
            except:
                pass
                
        return result_data

pipeline_thinking = IntegratedVLMGroundingThinking(device="cuda")

Initializing Qwen3-VL 8B Thinking Pipeline on NVIDIA RTX A6000


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Loading SAM 3...


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

Pipeline Ready.


How the Coordinate System is Defined:  
The pipeline uses the standard computer vision coordinate system (used by OpenCV, PIL, and SAM):  
Origin $(0, 0)$: The Top-Left corner of the image.  
X-Axis: Increases to the Right.  
Y-Axis: Increases Downwards.

In [10]:
import json
import os
from tqdm import tqdm

OUTPUT_FILE = os.path.join(OUTPUT_DIR, "pipeline_predictions.json")
results_list = []

print(f"Starting evaluation on {len(dataset)} images...")

for i in tqdm(range(len(dataset))):
    item = dataset[i]
    img_path = item['image_path']
    img_id = item['image_id']
    gt_label = item['label']
    
    anno_filename = img_id.replace('.png', '.json').replace('.jpg', '.json')
    anno_path = os.path.join(ANNOTATION_DIR, anno_filename)
    
    query = "Find the {gt_label}" 
    if os.path.exists(anno_path):
        with open(anno_path, 'r') as f:
            data = json.load(f)
            # Extract the first referring sentence available
            if 'objects' in data and len(data['objects']) > 0:
                query = data['objects'][0].get('referring_sentence', query)

    result = pipeline_thinking.process_query(
        image_path=img_path, 
        complex_query=query, 
        image_id=img_id, 
        known_category=gt_label 
    )
    if result:
        #print(result.get('raw_response', "No response (Pipeline failed early)"))
        print(result.get('selection_prompt', "No response (Pipeline failed early)"))
        results_list.append(result)
    

with open(OUTPUT_FILE, "w") as f:
    json.dump(results_list, f, indent=4)

print(f"Saved predictions to {OUTPUT_FILE}")

Starting evaluation on 2 images...


 50%|█████     | 1/2 [00:00<00:00,  1.60it/s]

No response (Pipeline failed early)


100%|██████████| 2/2 [01:59<00:00, 59.58s/it]

The user query is: 'The small vehicle is located at the bottom-right corner.'.

I have mapped the image to a 512x512 coordinate grid.
COORDINATE SYSTEM CONVENTION:
- Origin (0,0) is the TOP-LEFT corner.
- X increases to the RIGHT (Max X=512).
- Y increases DOWNWARDS (Max Y=512).
- Center of the image is approx (256, 256).

Here are the detected 'vehicle' candidates:
ID 1: Center=(220, 328), Size=(21w x 10h), Area=208, Angle=42.0
ID 2: Center=(325, 260), Size=(19w x 11h), Area=202, Angle=45.0
ID 3: Center=(449, 504), Size=(46w x 19h), Area=877, Angle=22.6
ID 4: Center=(498, 399), Size=(28w x 11h), Area=310, Angle=36.9
ID 5: Center=(198, 331), Size=(23w x 13h), Area=306, Angle=31.0
ID 6: Center=(433, 330), Size=(19w x 12h), Area=229, Angle=45.0

Identify the ID that matches the query based on location and size (Area). Think step-by-step. Available IDs: 1, 2, 3, 4, 5, 6.
Final Answer: <ID>
Saved predictions to eval_results/pipeline_predictions.json


In [6]:
with open(OUTPUT_FILE, "w") as f:
    json.dump(results_list, f, indent=4)

print(f"Saved predictions to {OUTPUT_FILE}")

Saved predictions to eval_results/pipeline_predictions.json


In [ ]:
test_img_path = dataset[5]['image_path'] 
user_query = "which is the airplane farthest from all other airplanes"
best_obb, ann_img = pipeline_thinking.process_query(test_img_path, user_query)

if best_obb:
    final_img = Image.open(test_img_path).convert("RGB")
    draw = ImageDraw.Draw(final_img)

    box_points = cv2.boxPoints(best_obb)
    box_points = np.int32(box_points)
    draw.polygon([tuple(pt) for pt in box_points], outline="#00FF00", width=5)
    
    plt.figure(figsize=(12, 12))
    plt.imshow(final_img)
    plt.title(f"Result: {user_query}")
    plt.axis("off")
    plt.show()


In [ ]:
import numpy as np
import cv2
import os
import json
from PIL import Image

def calculate_iou_poly(pred_points, gt_points):

    box1 = np.array(pred_points, dtype=np.float32)
    box2 = np.array(gt_points, dtype=np.float32)

    # Calculate Area of both polygons
    area1 = cv2.contourArea(box1)
    area2 = cv2.contourArea(box2)
    
    try:
        intersection_area, _ = cv2.intersectConvexConvex(box1, box2)
    except Exception as e:
        return 0.0
        
    union_area = area1 + area2 - intersection_area
    
    if union_area <= 0: return 0.0
    
    iou = intersection_area / union_area
    return iou

total_iou = 0
correct_counts_50 = 0 # IoU >= 0.5
correct_counts_25 = 0 # IoU >= 0.25
total_samples = 0

with open(os.path.join(OUTPUT_DIR, "pipeline_predictions.json"), 'r') as f:
    preds = json.load(f)

print(f"Comparing {len(preds)} predictions to Ground Truth...")

for pred in preds:
    if pred.get('predicted_obb') is None:
        continue
    
    image_id = pred['image_id']
    
    anno_filename = image_id.replace('.png', '.json').replace('.jpg', '.json')
    anno_path = os.path.join(ANNOTATION_DIR, anno_filename)
    
    if os.path.exists(anno_path):
        with open(anno_path, 'r') as f:
            gt_data = json.load(f)
            
        # We need the image dimensions to Un-Normalize the GT coordinates
        # Try to load image to get (width, height)
        img_path = os.path.join(IMAGE_DIR, image_id)
        if not os.path.exists(img_path):
             # Try replacing png with jpg if needed
             img_path = img_path.replace('.png', '.jpg')
             
        if os.path.exists(img_path):
            with Image.open(img_path) as img:
                img_w, img_h = img.size
        else:
            # Skip if we can't find image dimensions
            continue

        # Extract Ground Truth Object
        # Assuming we compare against the first object in the list as the target
        if 'objects' in gt_data and len(gt_data['objects']) > 0:
            gt_obj = gt_data['objects'][0]
            
            # 'obj_corner' is [x1, y1, x2, y2, x3, y3, x4, y4] (normalized 0-1)
            raw_corners = gt_obj['obj_corner']
            
            gt_poly_abs = []
            # Iterate pairwise (x, y)
            for k in range(0, 8, 2):
                # Scale Normalized X by Image Width
                x_abs = raw_corners[k] * img_w
                # Scale Normalized Y by Image Height
                y_abs = raw_corners[k+1] * img_h
                
                gt_poly_abs.append([x_abs, y_abs])
            
            # Now both pred['predicted_obb'] and gt_poly_abs are in ABSOLUTE pixels
            iou = calculate_iou_poly(pred['predicted_obb'], gt_poly_abs)
            
            total_iou += iou
            if iou >= 0.50: correct_counts_50 += 1
            if iou >= 0.25: correct_counts_25 += 1
            total_samples += 1

if total_samples > 0:
    print(f"\n--- Evaluation Results ---")
    print(f"Total Samples Evaluated: {total_samples}")
    print(f"Average IoU: {total_iou / total_samples:.4f}")
    print(f"Accuracy (IoU >= 0.50): {correct_counts_50 / total_samples * 100:.2f}%")
    print(f"Accuracy (IoU >= 0.25): {correct_counts_25 / total_samples * 100:.2f}%")
else:
    print("No valid comparisons made. Check paths or data alignment.")

In [ ]:

# ans = pipeline_thinking.extract_target_class_heuristic("The small vehicle is located at the bottom-right corner")
# print(ans)